# Fundamentals of Data Science - Network Science Lab

# Lab Session 01 - Networks

In [ ]:
import igraph as ig
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base_dir = Path.cwd()

vertex_output_size = 30

## Defining Networks

A network can be seen as a complex system whose components interact according to three main properties:
1. **Adaptation:** The components of the system mutate and self-organize in order to maintain certain properties.
2. **Non-linearity:** A change in the input is not proportional to a change in the output.
3. **Emergence:** The system as a whole shows some properties that the individual components do not possess.

A network can be formally represented as a graph $G = (V, E)$ that is described using a set of nodes $V$ and a set of edges $E$ that connect pairs of vertices in the graph.

In particular, a graph is said to be undirected if its edges indicate unordered pairs, which means that
$$E = \{\{u, v\}: u, v \in V, u \neq v\}$$
By contrast, a graph is said to be directed if its edges represent ordered pairs, which means that
$$E = \{(u, v): u, v \in V, u \neq v\}$$

### Graphs as Edge Lists/Dataframes

In [ ]:
# Edges stored as a Python list of tuples:
# - Each tuple (u, v) represents an edge between vertices u and v.
# - Tuples are immutable, ensuring edges are not accidentally modified.
# - Using 0-based indexing here because igraph (Python) uses 0-based vertex IDs.
edges = [(0,1), (0,2), (1,2), (2,3), (3,4), (3,5), (4,5)]

# Create an undirected graph from the edge list.
# igraph.Graph stores:
# - Vertex set V: Implicitly indexed 0, ..., n-1.
# - Edge set E: Stored internally in an efficient adjacency structure.
g_py = ig.Graph(edges=edges, directed=False)

print(g_py)

In [ ]:
# Edge list as a DataFrame.
# - Each row represents one edge.
# - Columns: Source and target vertex labels.
# - DataFrame is ideal for tabular manipulation, filtering, exporting.
edge_list_py = pd.DataFrame({
    "source": [0, 0, 1, 2, 3, 3, 4],
    "target": [1, 2, 2, 3, 4, 5, 5]
})

print(edge_list_py)

# Create graph from DataFrame.
# - Graph.DataFrame expects a DataFrame of edges and optional vertex DataFrame.
# - vertex IDs in DataFrame can be integers or strings.
g_py = ig.Graph.DataFrame(edge_list_py, directed=False)

print(g_py)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_py,
    target=ax, #plot on the axes
    vertex_label=list(range(0, g_py.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

### Graphs as Adjacency Matrices

In [ ]:
# Adjacency Matrix written explicitly (1 = edge, 0 = no edge) as a list of lists.
adj_matrix = [
    [0, 1, 1, 0, 0, 0],  # Vertex 0.
    [1, 0, 1, 0, 0, 0],  # Vertex 1.
    [1, 1, 0, 1, 0, 0],  # Vertex 2.
    [0, 0, 1, 0, 1, 1],  # Vertex 3.
    [0, 0, 0, 1, 0, 1],  # Vertex 4.
    [0, 0, 0, 1, 1, 0]   # Vertex 5.
]

# Create igraph from adjacency matrix.
g_from_matrix = ig.Graph.Adjacency(adj_matrix, mode=ig.ADJ_UNDIRECTED)

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_from_matrix,
    target=ax,
    vertex_label=list(range(0, g_py.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

### Graphs as Adjacency Lists

In [ ]:
# Adjacency List written explicitly as a list of lists.
adj_list = [
    [1, 2],     # Vertex 0.
    [0, 2],     # Vertex 1.
    [0, 1, 3],  # Vertex 2.
    [2, 4, 5],  # Vertex 3.
    [3, 5],     # Vertex 4.
    [3, 4]      # Vertex 5.
]


# Since igraph does not have a function to create a graph from an adjacency list, it is converted to an edge list or adjacency matrix for plotting.
edge_list = []
for i in range(0, len(adj_list)):
  for j in adj_list[i]:
    if i<j:
      edge_list.append((i,j))

print(edge_list)

adj_matrix = []

for i in range(0, len(adj_list)):
    tmp_list = [0] * 6
    for j in adj_list[i]:
        tmp_list[j] = 1
    adj_matrix.append(tmp_list)

print(adj_matrix)

## Neighbourhoods and Subgraphs

An adjacent vertex of a node $v$ in a graph is any vertex that is connected to $v$ by an edge.

In particular, the neighbourhood of a vertex $v$ in a graph $G$ is a subgraph of $G$ that contains all the vertices adjacent to $v$ and the edges connecting $v$ to its neighbours.

Additionally, the neighbourhood of a vertex $v$ is said to be open, if it does not include $v$, or closed, if it includes $v$ as well.

In [ ]:
print(g_py.neighborhood(3))

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_py.subgraph(g_py.neighborhood(3)),
    target=ax,
    vertex_label=sorted(g_py.neighborhood(3)),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

Let $G = (V, E)$ be any graph: for a set $S \subseteq V$, the subgraph $G[S]$ is the graph whose vertex set is $S$ and whose edge set contains all the edges in $E$ that have both endpoints in $S$.

In [ ]:
g_py_sg = g_py.induced_subgraph([0,1,4,5])

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_py_sg,
    target=ax,
    vertex_label= [0,1,4,5],
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

## Common Graphs

### Cliques

A connected graph $G = (V, E)$ is said to be a "clique" if it has the maximum possible number of edges.

For a clique with $|V| = n$ nodes, it holds that $|E| = \binom{n}{2} = \frac{n(n - 1)}{2}$.

In [ ]:
clique = ig.Graph.Full(n=5) # No need to specify the number of edges because every node will be connected.

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    clique,
    target=ax,
    vertex_label=list(range(0, clique.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

### Trees

A connected graph $G(V, E)$ is said to be a "tree" if it has the minimum possible number of edges.

For a tree with $|V| = n$ nodes, it holds that $|E| = n - 1$.

In [ ]:
tree = ig.Graph.Tree(n=20, children=2)

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    tree,
    target=ax,
    vertex_label=list(range(0, tree.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

In [ ]:
line_tree = ig.Graph.Tree(n=5, children=1) # Setting 1 child node creates a line tree graph.

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    line_tree,
    target=ax,
    vertex_label=list(range(0, line_tree.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

## Graph Measures

In [ ]:
# Create a graph containing self-loops for each node.
not_simple_g = ig.Graph(edges = [(0,0),(0,1),(1,0),(1,1),(1,2),(2,1),(2,2)])

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    not_simple_g,
    target=ax, #plot on the axes
    vertex_label=list(range(0, not_simple_g.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2,
    edge_color = ["red","grey","grey","red","grey","grey","red"]
)

# The `simplify` function allows to eliminate self-loops and repeated links in a graph.
simplified_g = not_simple_g.copy()
simplified_g.simplify()

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    simplified_g,
    target=ax,
    vertex_label=list(range(0, simplified_g.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

### Density

The density of a graph $G = (V, E)$ measures the number of edges compared to the maximum possible number of edges.

- If $G = (V, E)$ is an undirected graph, then
  $$\delta = \frac{|E|}{|E_{\max}|} = \frac{|E|}{\binom{|V|}{2}} = \frac{2|E|}{|V|(|V| - 1)}$$
- If $G = (V, E)$ is a directed graph, then
  $$\delta = \frac{|E|}{|E_{\max}|} = \frac{|E|}{2\binom{|V|}{2}} = \frac{|E|}{|V|(|V| - 1)}$$

Most real-world networks are sparse because, even though they are composed by many nodes, their density tends to be close to zero.

In [ ]:
g_py

# Density measurement using igraph.
print("Graph density: ",round(g_py.density(), ndigits = 2))

# Custom implementation.
def density_manual(g):
  density = (2*g.ecount())/(g.vcount()*(g.vcount()-1))
  density = round(density, ndigits = 2)
  return density

print("Graph density with our function: ",density_manual(g_py))

### Degree

The degree of a node denotes the number of connections of a node.

- If $G = (V, E)$ is an undirected graph, then
  $$k_i = \sum_{j = 1}^{|V|} A_{ij} \text{, with } \sum_i k_i = 2|E|$$
  More specifically, the mean degree of an undirected graph can be formulated as
  $$\overline{k} = \frac{1}{|V|}\sum_i k_i = \frac{2|E|}{|V|}$$
  Furthermore, if a network is simple, meaning that there are no self-loops or multiple edges, each node can have at most $k_{\max} = n - 1$ connections.
- If $G = (V, E)$ is a directed graph, then it is necessary to make a distinction between the in-degree, which denotes the number of incoming edges, and the out-degree, which denotes the number of outgoing edges.
  \begin{align*}
  k_i^{in} &= \sum_j A_{ji} \\
  k_i^{out} &= \sum_j A_{ij} \\
  k_i^{tot} &= k_i^{in} + k_i^{out} \text{, with } \sum_i k_i^{tot} = 2|E|
  \end{align*}
  More specifically, the mean degree of a directed graph can be formulated as
  $$\overline{k^{in}} = \frac{1}{|V|}\sum_i k_i^{in} = \frac{|E|}{|V|} = \frac{1}{|V|}\sum_i k_i^{out} = \overline{k^{out}}$$

In [ ]:
directed_edges = [(0,1), (1,2), (2,0), (2,3), (3,4), (4,5), (5,3)]

g_py_directed = ig.Graph(edges = directed_edges, directed = True)

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    g_py_directed,
    target=ax,
    vertex_label=list(range(0, g_py_directed.vcount())),
    vertex_size=vertex_output_size,
    vertex_color="lightblue",
    edge_width=2
)

print(g_py_directed.get_adjacency())

## Weighted Networks

Given a graph $G = (V, E)$, it is possible to add attributes to its nodes, typically to add information about the information they represent, and its edges, typically to specify the importance of each edge.

In [ ]:
# Assign labels and colors as vertex attributes.
g_py.vs["label"] = ["A","B","C","D","E","F"]
g_py.vs["color"] = ["blue","blue","blue","red","red","red"]

# Inspect attributes.
print("Vertex labels:", g_py.vs["label"])
print("Vertex colors:", g_py.vs["color"])

fig, ax = plt.subplots()
ig.plot(
    g_py,
    target=ax,
    vertex_label= g_py.vs["label"],
    vertex_label_color = "white",
    vertex_size=vertex_output_size,
    vertex_color=g_py.vs["color"],
    edge_width=2
)

In [ ]:
# Assign weights as edge attributes.
g_py.es["weight"] = [10, 5, 10, 1, 5, 10, 5]

fig, ax = plt.subplots()
ig.plot(
    g_py,
    target=ax,
    vertex_label= g_py.vs["label"],
    vertex_label_color = "white",
    vertex_size=vertex_output_size,
    vertex_color=g_py.vs["color"],
    edge_width=g_py.es["weight"]
)

Generally speaking, a weighted network is defined as a graph $G = (V, E, w)$, where the function $w: E \to \mathbb{R}^+$ assigns a (positive) weight to each edge.

In particular, adjacency matrix of a weighted network can be defined as
$$W_{ij} = \begin{cases}
  w_{ij} &\quad\text{if $(i, j) \in E$} \\
  0 &\quad\text{otherwise} \\
\end{cases}$$
Note that, whenever a weighted network is undirected, its adjacency matrix is always symmetric, although, if a weighted network is directed, its adjacency matrix may or may not be symmetric.

Furthermore, the strength of a node represents a generalization of the degree of a node for weighted networks.
- If $G = (V, E, w)$ is an undirected weighted network, then
  $$s_i = \sum_j w_{ij}$$
- If $G = (V, E, w)$ is a directed weighted network, then it is necessary to make a distinction between the in-strength and the out-strength.
  \begin{align*}
  s_i^{in} &= \sum_j w_{ji} \\
  s_i^{out} &= \sum_j w_{ij}
  \end{align*}

In [ ]:
fig, ax = plt.subplots()
ig.plot(
    g_py,
    target=ax,
    vertex_label= g_py.vs["label"],
    vertex_label_color = "white",
    vertex_size= [x*3 for x in g_py.strength(weights = "weight")],
    vertex_color=g_py.vs["color"],
    edge_width=g_py.es["weight"]
)

print("The strength values are:", g_py.strength(weights = "weight"))

print(
  "The weighted adjacency matrix is: \n",
  g_py.get_adjacency(attribute = "weight"))

## Real Networks

In [ ]:
zkk = ig.Graph.Read_GraphML(
  str(base_dir.parent / "Datasets" / "zachary_karate_club.graphml"))

print(zkk.vertex_attributes())
zkk.vs["color"] = ["blue" if val == 1 else "red" for val in zkk.vs["color"]]

print(zkk.is_directed())

fig, ax = plt.subplots(figsize=(5, 4))
ig.plot(
    zkk,
    target=ax,
    vertex_label= zkk.vs["name"],
    vertex_label_color = "black",
    vertex_size= vertex_output_size/3,
    vertex_color=zkk.vs["color"],
    vertex_frame_color='white',
    edge_color= "grey"
)

print("Number of nodes", zkk.vcount())
print("Number of edges", zkk.ecount())

# Network density.
def network_density(n,m,directed):
  if n<=1:
    return 0

  if directed:
    density = m/(n*(n-1))
  else:
    density = (2*m)/(n*(n-1))

  return density

print("Custom density:",
network_density(zkk.vcount(),zkk.ecount(),directed=False))
print("Built-in density:", zkk.density())

# Degree distribution.
fix, ax = plt.subplots(figsize=(5, 4))

ax.hist(zkk.degree(),
        bins = range(0,max(zkk.degree())+1),
        color = "red",
        edgecolor = "white")

ax.set_title("Frequency of Degree (Histograms)")
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.set_xticks(range(0,max(zkk.degree())+1))

plt.show()

print(zkk.degree())

# It is possible to count degree frequency using a Pandas series.
degree_frequency = pd.Series(zkk.degree())
print("Degree values \n",
degree_frequency)

degree_frequency = pd.Series(zkk.degree()).value_counts() # Remove values for which the degree frequency is zero.
print("Degree frequencies \n",
degree_frequency)

degree_frequency = degree_frequency.sort_index() # Sort by degree value.
print("Degree frequencies with missing degree values \n", degree_frequency)

The degree distribution of a network describes the frequency of vertex degrees within the graph.

Being a discrete distribution, the degree distribution of a graph is normally studied using raw frequencies or by defining a probability mass function.

Alternatively, when working on large networks, it is possible to approximate the degree distribution using a (complementary) cumulative distribution function, which computes (the complement of) tail probabilities for the degree distribution.

In [ ]:
# Define the full range of integers.
full_range = range(min(zkk.degree()), max(zkk.degree())+1)

# Re-index the frequency series to include missing numbers.
degree_frequency = degree_frequency.reindex(full_range, fill_value=0)
print("Degree frequencies WITHOUT missing degree values \n", degree_frequency)

normalized_degree_frequency = degree_frequency/zkk.vcount()
print("The normalized degree values sum up to: ",
sum(normalized_degree_frequency))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))

In [ ]:
# Plot the degree frequencies.
ax[0].scatter(degree_frequency.index,
degree_frequency.values, color = "red")
ax[0].set_title("Frequency of Elements (Scatter Plot)")
ax[0].set_xlabel("Degree")
ax[0].set_ylabel("Frequency")
ax[0].set_box_aspect(1)

# Plot the probability mass function.
ax[1].scatter(normalized_degree_frequency.index,
normalized_degree_frequency.values, color = "red")
ax[1].set_title("Probability of Elements (Scatter Plot)")
ax[1].set_xlabel("Degree")
ax[1].set_ylabel("Probability (PMF)")
ax[1].set_box_aspect(1)

plt.show()

In [ ]:
# Plot the cumulative distribution function.
ax[0].scatter(normalized_degree_frequency.index,
normalized_degree_frequency.cumsum().values,
color = "red")
ax[0].set_title("Probability of Elements")
ax[0].set_xlabel("Degree")
ax[0].set_ylabel("CDF")
ax[0].set_box_aspect(1)

# Plot the complementary cumulative distribution function.
ax[1].scatter(normalized_degree_frequency.index,
1-normalized_degree_frequency.cumsum().values,
color = "red")
ax[1].set_title("Probability of Elements")
ax[1].set_xlabel("Degree")
ax[1].set_ylabel("CCDF")
ax[1].set_box_aspect(1)

plt.show()